In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

train = pd.read_parquet("../data/processed/train_woe.parquet")
test  = pd.read_parquet("../data/processed/test_woe.parquet")

X_train, y_train = train.drop(columns="target"), train["target"]
X_test,  y_test  = test.drop(columns="target"),  test["target"]
print(X_train.shape, X_test.shape)

(306462, 10) (283026, 10)


In [2]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# Pairwise correlation between WoE features
display(X_train.corr().round(2))

# VIF: values above ~5 suggest a feature is redundant
Xc = add_constant(X_train.dropna())
vif = pd.DataFrame({
    "feature": X_train.columns,
    "vif": [variance_inflation_factor(Xc.values, i + 1) for i in range(X_train.shape[1])],
})
vif.sort_values("vif", ascending=False)

,home_ownership,annual_inc,purpose,dti,inq_last_6mths,revol_util,mort_acc,credit_history_yrs,fico_avg,loan_to_income
home_ownership,1.00,0.28,0.06,0.01,-0.06,0.00,0.63,0.24,0.14,0.07
annual_inc,0.28,1.00,0.03,0.23,-0.07,-0.05,0.38,0.28,0.15,0.36
purpose,0.06,0.03,1.00,0.01,0.03,-0.04,0.05,0.03,0.02,-0.06
dti,0.01,0.23,0.01,1.00,0.01,0.17,0.05,-0.04,0.07,0.22
inq_last_6mths,-0.06,-0.07,0.03,0.01,1.00,-0.08,-0.08,-0.00,0.07,-0.09
revol_util,0.00,-0.05,-0.04,0.17,-0.08,1.00,0.01,-0.00,0.48,0.06
mort_acc,0.63,0.38,0.05,0.05,-0.08,0.01,1.00,0.37,0.12,0.12
credit_history_yrs,0.24,0.28,0.03,-0.04,-0.00,-0.00,0.37,1.00,0.11,0.04
fico_avg,0.14,0.15,0.02,0.07,0.07,0.48,0.12,0.11,1.00,-0.01
loan_to_income,0.07,0.36,-0.06,0.22,-0.09,0.06,0.12,0.04,-0.01,1.00


,feature,vif
6,mort_acc,1.910084
0,home_ownership,1.678659
1,annual_inc,1.481727
5,revol_util,1.421833
8,fico_avg,1.418776
7,credit_history_yrs,1.211948
9,loan_to_income,1.203977
3,dti,1.131592
4,inq_last_6mths,1.045501
2,purpose,1.012611


In [3]:
# Plain logistic regression on WoE features (light regularisation)
model = LogisticRegression(C=10, max_iter=1000)
model.fit(X_train, y_train)

coefs = pd.Series(model.coef_[0], index=X_train.columns).sort_values()
print("intercept:", model.intercept_[0].round(3))
coefs

intercept: -1.881


inq_last_6mths       -1.084832
purpose              -0.977511
fico_avg             -0.848553
loan_to_income       -0.760068
dti                  -0.625969
annual_inc           -0.443942
home_ownership       -0.355027
credit_history_yrs   -0.279861
mort_acc             -0.252735
revol_util           -0.113502
dtype: float64

In [4]:
def ks_stat(y_true, y_score):
    # KS = maximum gap between the cumulative bad and good distributions
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return (tpr - fpr).max()

for name, X, y in [("train", X_train, y_train), ("test", X_test, y_test)]:
    p = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, p)
    print(f"{name}: AUC={auc:.3f}  Gini={2*auc-1:.3f}  KS={ks_stat(y, p):.3f}")

train: AUC=0.652  Gini=0.304  KS=0.219
test: AUC=0.654  Gini=0.307  KS=0.221


In [5]:
p_test = model.predict_proba(X_test)[:, 1]
cal = pd.DataFrame({"pred": p_test, "actual": y_test.values})
cal["decile"] = pd.qcut(cal["pred"], 10, labels=False, duplicates="drop") + 1

# Compare predicted PD with observed bad rate in each risk decile
cal.groupby("decile").agg(avg_pred=("pred", "mean"),
                          bad_rate=("actual", "mean"),
                          n=("actual", "size")).round(3)

,avg_pred,bad_rate,n
decile,,,
1,0.043,0.041,28304
2,0.067,0.076,28302
3,0.084,0.098,28302
4,0.099,0.118,28303
5,0.114,0.132,28302
6,0.130,0.154,28303
7,0.148,0.171,28302
8,0.169,0.195,28303
9,0.198,0.227,28302
